# Spotify Data Dashboard (Part 2)

Welcome to the Spotify Data Dashboard! This interactive tool, built using the Spotipy library, lets you dive into the world of music with data directly from Spotify. Whether you're exploring top hits, discovering more about your favorite artists, analyzing your recent listening habits, or uncovering popular music genres, this dashboard makes learning about music data both fun and educational.

### What You'll Discover:

* **Top Tracks:** Find out which songs are currently making waves around the world. You’ll see which tracks are the most popular and learn about their artists.

* **Artist Searches:** Have a favorite artist? Search for them and get detailed insights, including their top tracks and related artists. This is a great way to find new music similar to what you already love.

* **Recent Tracks:** Look back at your own Spotify listening history. What songs do you play the most? Are there patterns based on the time of day? Analyzing your personal music trends can be a good reflection of your musical tastes.

* **Top Genres:** Music comes in many styles and genres. Discover which genres are the most popular.

Each section of this dashboard not only provides valuable insights but also includes visual representations like charts and graphs, making the data easy to understand and visually appealing.

## Project Setup

In [1]:
# Install the spotipy and streamlit library
!pip install spotipy streamlit -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 32.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 2.4 MB/s eta 0:00:00


In [2]:
# Import libraries for spotipy, dataframe manipulation, and visualisation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import streamlit as st
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials, SpotifyOAuth
import spotify_boilerplate as mysp

In [3]:
### AUTHENTICATION ###

# Authentication setup for accessing Spotify API
def setup_spotify_auth(personal=True):
    """Initialize Spotify client with either personal or client credentials based on the `personal` flag."""
    if personal:
        auth_manager = SpotifyOAuth(
            client_id=YOUR_CLIENT_ID,
            client_secret=YOUR_CLIENT_SECRET,
            redirect_uri="http://localhost:8888/callback",
            scope="user-top-read user-read-recently-played",
            open_browser=False
        )
        return spotipy.Spotify(auth_manager=auth_manager)
    else:
        client_credentials_manager = SpotifyClientCredentials(
            client_id=YOUR_CLIENT_ID,
            client_secret=YOUR_CLIENT_SECRET
        )
        return spotipy.Spotify(client_credentials_manager=client_credentials_manager)


# Connect to Spotify
personal = True
YOUR_CLIENT_ID = '4fea4c290c8e4c99b4a926acf7296d9a'
YOUR_CLIENT_SECRET = 'fd2c5dead1a34e628cd19dd122834122'

sp = setup_spotify_auth(personal=personal)

In [4]:
top_tracks = mysp.fetch_top_tracks_user(sp, limit=10)

Go to the following URL: https://accounts.spotify.com/authorize?client_id=4fea4c290c8e4c99b4a926acf7296d9a&response_type=code&redirect_uri=http%3A%2F%2Flocalhost%3A8888%2Fcallback&scope=user-top-read+user-read-recently-played
Enter the URL you were redirected to: http://localhost:8888/callback?code=AQAiKH5U4jesefYGg-RC6vmozt95TCATU_Abj-vT33SmC_-SsUD4kTTpRV9vsZDNRp4iyE4E-SXMnmiFeZvm614wXNDgvdse2VYd0FznP2bt5V1UFax9dlhI52omvcbCCXKUJCZza91Cl2Z_tn8MdOk9zZufmvspx4ZP8ioE480Tq_yAENP_GzajN4kVo0SoHmXzEP8y2HGFKvovjX-jsT3d0WKhiJfexKiJrg



## STREAMLIT APP CODE

### 1. Libraries

In [5]:
%%writefile sp_dashboard.py
# Import libraries for spotipy, dataframe manipulation, and visualisation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import streamlit as st
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials, SpotifyOAuth
import spotify_boilerplate as mysp

Writing sp_dashboard.py


### 2. Authentication

In [6]:
%%writefile -a sp_dashboard.py

### AUTHENTICATION ###

# Authentication setup for accessing Spotify API
def setup_spotify_auth(personal=True):
    """Initialize Spotify client with either personal or client credentials based on the `personal` flag."""
    if personal:
        auth_manager = SpotifyOAuth(
            client_id=YOUR_CLIENT_ID,
            client_secret=YOUR_CLIENT_SECRET,
            redirect_uri="http://localhost:8888/callback",
            scope="user-top-read user-read-recently-played",
            open_browser=False
        )
        return spotipy.Spotify(auth_manager=auth_manager)
    else:
        client_credentials_manager = SpotifyClientCredentials(
            client_id=YOUR_CLIENT_ID,
            client_secret=YOUR_CLIENT_SECRET
        )
        return spotipy.Spotify(client_credentials_manager=client_credentials_manager)


# Connect to Spotify
personal = True
YOUR_CLIENT_ID = '4fea4c290c8e4c99b4a926acf7296d9a'
YOUR_CLIENT_SECRET = 'fd2c5dead1a34e628cd19dd122834122'

sp = setup_spotify_auth(personal=personal)

Appending to sp_dashboard.py


### 3. Select Data & Visualise

Top Tracks & Artists

In [7]:
%%writefile -a sp_dashboard.py

def top_barplot(top_names):
  top_names = top_names.sort_values(by='popularity', ascending=False)  # optionally sort
  colour = 'viridis'  # You can choose different color palettes like 'viridis', 'rocket', etc.

  ### PLOT ###

  sns.set(style="whitegrid")  # Set the aesthetic style of the plots
  fig, ax = plt.subplots(figsize=(10, 8))  # Create a figure and an axis


  bar_plot = sns.barplot(
      x='popularity',  # Make sure this corresponds to your column name
      y='name',  # Make sure this corresponds to your column name
      data=top_names,
      palette=colour,
      ax=ax  # Pass the axis to seaborn
  )

  # Set the title and labels
  ax.set_xlabel('Popularity')
  ax.set_ylabel('')
  st.pyplot(fig)

Appending to sp_dashboard.py


Related Artists

In [8]:
%%writefile -a sp_dashboard.py

def plot_related_artists(sp, artist_name):

  similar_artists = mysp.fetch_related_artists(sp, artist_name)
  data = similar_artists.head(10) # optionally filter (e.g. only 10 artists)
  colour = 'lightblue'  # You can choose different colors like 'coral', 'green', etc.

  ### PLOT ###

  G = nx.Graph()

  G.add_node(artist_name)  # Add the main artist as a node

  # Add related artists
  for index, artist in data.iterrows():
      G.add_node(artist['name'])  # Add related artist
      G.add_edge(artist_name, artist['name'])  # Create an edge between main artist and related artist

  fig, ax = plt.subplots(figsize=(8, 8))
  pos = nx.spring_layout(G, seed=7)  # Positions for all nodes
  nx.draw(G, pos, with_labels=True, node_color=colour, edge_color='gray')
  st.pyplot(fig)

Appending to sp_dashboard.py


Audio Features

In [9]:
%%writefile -a sp_dashboard.py

def radar_chart(df):
  # Create a radar chart
  labels = np.array(df.columns.to_list())
  num_vars = len(labels)

  # Create angle for each axis
  angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
  angles += angles[:1]  # complete the loop

  # Plot each track
  fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
  for i, row in df.iterrows():
      data = row.tolist()
      data += data[:1]  # complete the loop
      ax.fill(angles, data, alpha=0.25)
      ax.plot(angles, data, label=i)  # Add each track's plot

  # Improve aesthetics
  ax.set_theta_offset(np.pi / 2)
  ax.set_theta_direction(-1)
  ax.set_rlabel_position(0)

  # Draw one axe per variable and add labels
  plt.xticks(angles[:-1], labels)

  # Add legend
  plt.legend(title='Tracks', loc='upper right', bbox_to_anchor=(1.1, 1.1))
  st.pyplot(fig)

Appending to sp_dashboard.py


### 4. Dashboard

In [10]:
%%writefile -a sp_dashboard.py

# Streamlit app
st.title('Magdalena\'s Spotify Dashboard')

# Sidebar for navigation
st.sidebar.title("Navigation")
options = st.sidebar.radio("Choose a view", ["Top Tracks", "Top Artists", "Related Artists", "Audio Features"])

# Slider for selecting number of top tracks
n_tracks = st.sidebar.slider('Select number of top songs to visualize', 1, 10, 5)
n_artists = st.sidebar.slider('Select number of top artists to visualize', 1, 10, 5)

# Top Tracks
if options == "Top Tracks":
  top_tracks = mysp.fetch_top_tracks_user(sp, limit=n_tracks)
  top_barplot(top_tracks)

# Top Artists
elif options == "Top Artists":
  top_artists = mysp.fetch_top_artists_user(sp, limit=n_artists)
  top_barplot(top_artists)

# Related Artists
elif options == "Related Artists":

  # Fetch top artists to populate the dropdown menu
  top_artists = mysp.fetch_top_artists_user(sp, limit=n_artists)

  # Dropdown menu for selecting an artist
  selected_artist = st.selectbox('Select an artist to view related artists', top_artists['name'])

  # Text input for manual artist entry
  manual_artist = st.text_input('Or type an artist name manually')

  if manual_artist:
      selected_artist = manual_artist

  if selected_artist:
      plot_related_artists(sp, selected_artist)

# Audio Features
elif options == "Audio Features":
    top_tracks = mysp.fetch_top_tracks_user(sp, limit=n_tracks)
    features_df = mysp.fetch_audio_features_for_tracks(sp, top_tracks)
    st.write(features_df)

    selected_features = st.multiselect(
        'Select features for the radar chart',
        options=features_df.columns.tolist(),
        default=['danceability', 'energy', 'acousticness']
    )

    if selected_features:
        df_selected = features_df[selected_features]
        radar_chart(df_selected)

else:
  st.write('Not implemented yet')

Appending to sp_dashboard.py


In [ ]:
!wget -q -O - ipv4.icanhazip.com

In [ ]:
!streamlit run sp_dashboard.py & npx localtunnel --port 8501




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.148.46.112:8501

npx: installed 22 in 6.723s
your url is: https://calm-bikes-sneeze.loca.lt
/content/sp_dashboard.py:51: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  bar_plot = sns.barplot(
